# Joint Transformer comparison — evaluator v2

**Fresh training is pending.** This notebook has been upgraded to the identical verified MASSIVE dataset and exact-character-span evaluator used by `main.ipynb`. All old outputs were archived and cleared because old scores and checkpoint selection used a different evaluator. No new performance result is claimed here.

The supplied experiment implements Stage 1 training plus development/OOV diagnostics. Stage 2 is defined and parameter-counted only; neither stage originally included final test evaluation. These limitations remain explicit. The model architectures and optimizer settings are preserved except the train-derived complete 111-tag BIO inventory.

Run from a fresh kernel with this filename in its working directory. Data are under `data/`; pretrained-vector cache is under `data/pretrained/`; new checkpoints are under ignored `artifacts/transformer_v2/`. Existing checkpoints are protected. The model receives only utterance-derived tokens/characters. Shared source is visible below; no repository module is imported at runtime.


In [ ]:
"""Tokenizer-independent scoring and pinned MASSIVE data shared by both notebooks.

This module intentionally uses only the Python standard library so its exact
source can be embedded in a standalone submission notebook. Scores are fractions.
Character offsets refer to the original Python Unicode string, with exclusive
ends. The recorded data digest pins the project's original release bytes; it is
not an independently published authenticity certificate.
"""

import hashlib
import json
import os
from pathlib import Path
import re
import tarfile
import tempfile
from urllib.request import Request, urlopen


EVALUATOR_VERSION = "exact_character_span_v2"
DATASET_SHA256 = "c70f75c6a543a26e249ec383df67733ad9b1066f6c0406c2e04a3f03356e407e"
EXPECTED_COUNTS = {"train": 11514, "dev": 2033, "test": 2974}
SOURCE_URL = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz"
ENGLISH_MEMBER = "1.0/data/en-US.jsonl"
_CONTRACT_ANNOTATION = re.compile(r"\[([A-Za-z_][A-Za-z_0-9]*)[ \t]*:[ \t]*(.*?)\]", re.DOTALL)
_CONTRACT_MAX_BYTES = 128 * 1024 * 1024


def _contract_json_hash(value):
    data = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


def parse_annotated_spans(utt, annot_utt):
    """Strip annotation syntax only, requiring exact original-text reconstruction."""
    if not isinstance(utt, str) or not isinstance(annot_utt, str):
        raise TypeError("Utterance and annotated utterance must be strings")
    pieces, spans = [], []
    annotation_cursor = plain_cursor = 0
    for match in _CONTRACT_ANNOTATION.finditer(annot_utt):
        before = annot_utt[annotation_cursor:match.start()]
        pieces.append(before)
        plain_cursor += len(before)
        text = match.group(2)
        if not text:
            raise ValueError("An annotated slot cannot have empty text")
        spans.append({"slot_type": match.group(1), "text": text,
                      "start": plain_cursor, "end": plain_cursor + len(text)})
        pieces.append(text)
        plain_cursor += len(text)
        annotation_cursor = match.end()
    pieces.append(annot_utt[annotation_cursor:])
    if "".join(pieces) != utt:
        raise ValueError("Removing annotation syntax does not exactly reconstruct utt")
    return spans


def load_verified_massive(path):
    """Load only the pinned original bytes, preserving every partition and row."""
    path = Path(path)
    if path.is_symlink() or not path.is_file():
        raise ValueError("Dataset must be a regular, non-symlink file")
    if not 0 < path.stat().st_size <= _CONTRACT_MAX_BYTES:
        raise ValueError("Dataset size is outside the accepted range")
    raw = path.read_bytes()
    observed_hash = hashlib.sha256(raw).hexdigest()
    if observed_hash != DATASET_SHA256:
        raise ValueError("Dataset SHA256 differs from the pinned original release; refusing reuse")
    splits = {name: [] for name in EXPECTED_COUNTS}
    seen = set()
    for number, line in enumerate(raw.decode("utf-8").splitlines(), 1):
        if not line.strip():
            raise ValueError(f"Blank dataset row at line {number}")
        row = json.loads(line)
        if not isinstance(row, dict):
            raise ValueError(f"Dataset row {number} is not an object")
        for field in ("id", "utt", "intent", "annot_utt", "partition", "locale"):
            if not isinstance(row.get(field), str) or (field != "annot_utt" and not row[field]):
                raise ValueError(f"Invalid dataset field {field!r} at line {number}")
        if row["locale"] != "en-US" or row["partition"] not in splits:
            raise ValueError(f"Unexpected locale or partition at line {number}")
        if row["id"] in seen:
            raise ValueError(f"Duplicate dataset ID: {row['id']!r}")
        seen.add(row["id"])
        parse_annotated_spans(row["utt"], row["annot_utt"])
        splits[row["partition"]].append(row)
    counts = {name: len(rows) for name, rows in splits.items()}
    if counts != EXPECTED_COUNTS:
        raise ValueError(f"Original split counts differ: {counts!r}")
    manifest = {
        "evaluator_version": EVALUATOR_VERSION,
        "dataset_sha256": observed_hash,
        "source_url": SOURCE_URL,
        "archive_member": ENGLISH_MEMBER,
        "counts": counts,
        "split_id_sha256": {name: _contract_json_hash([row["id"] for row in rows])
                            for name, rows in splits.items()},
        "split_content_sha256": {name: _contract_json_hash(rows)
                                 for name, rows in splits.items()},
        "identity_scope": "Pinned original bytes; original row order and every record field retained",
    }
    return splits, manifest


def ensure_verified_massive(path):
    """Reuse verified data or safely stream its exact member from the official URL.

    Existing mismatched files are never overwritten. A verified temporary file
    is published using an exclusive atomic hard link; archive-controlled names
    are never used for filesystem writes and extract/extractall are not used.
    """
    path = Path(path)
    if path.exists() or path.is_symlink():
        return load_verified_massive(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(prefix=".massive-verified-", suffix=".part", dir=path.parent)
    temporary = Path(temporary_name)
    try:
        found = False
        with os.fdopen(descriptor, "wb") as output:
            request = Request(SOURCE_URL, headers={"User-Agent": "SC4002-verified-comparison/2"})
            with urlopen(request, timeout=60) as response, tarfile.open(fileobj=response, mode="r|gz") as archive:
                for member in archive:
                    if member.name != ENGLISH_MEMBER:
                        continue
                    if not member.isfile() or not 0 < member.size <= _CONTRACT_MAX_BYTES:
                        raise ValueError("English archive member must be a bounded regular file")
                    source = archive.extractfile(member)
                    if source is None:
                        raise ValueError("English archive member could not be read")
                    written = 0
                    with source:
                        for block in iter(lambda: source.read(1024 * 1024), b""):
                            written += len(block)
                            if written > member.size:
                                raise ValueError("English archive member exceeds its declared size")
                            output.write(block)
                    if written != member.size:
                        raise ValueError("English archive member is truncated")
                    found = True
                    break
            output.flush()
            os.fsync(output.fileno())
        if not found:
            raise ValueError("Official archive does not contain its exact English member")
        verified = load_verified_massive(temporary)
        try:
            os.link(temporary, path)
        except FileExistsError:
            return load_verified_massive(path)
        return verified
    finally:
        temporary.unlink(missing_ok=True)


def _contract_field(value, key):
    if isinstance(value, dict):
        if key not in value:
            raise ValueError(f"Missing required field: {key}")
        return value[key]
    try:
        return getattr(value, key)
    except AttributeError as error:
        raise ValueError(f"Missing required field: {key}") from error


def _contract_index(records, source):
    indexed = {}
    for record in records:
        record_id = _contract_field(record, "record_id")
        intent = _contract_field(record, "intent")
        if not isinstance(record_id, str) or not record_id or not isinstance(intent, str) or not intent:
            raise ValueError(f"{source} record ID and intent must be nonempty strings")
        if record_id in indexed:
            raise ValueError(f"Duplicate {source} record ID: {record_id!r}")
        indexed[record_id] = record
    return indexed


def _contract_span_keys(record, utt):
    keys = set()
    for span in _contract_field(record, "slots"):
        slot_type, text, start, end = (_contract_field(span, name)
                                     for name in ("slot_type", "text", "start", "end"))
        if not isinstance(slot_type, str) or not slot_type or not isinstance(text, str) or not text:
            raise ValueError("Slot type and text must be nonempty strings")
        if type(start) is not int or type(end) is not int or not 0 <= start < end or end - start != len(text):
            raise ValueError("Slot character offsets must describe its exact nonempty text")
        if utt is not None and (end > len(utt) or utt[start:end] != text):
            raise ValueError("Slot offsets and text do not match the original utterance")
        key = (slot_type, start, end, text)
        if key in keys:
            raise ValueError("Duplicate identical slot occurrence")
        keys.add(key)
    return keys


def score_exact_spans(gold, predictions):
    """Score typed objects or dictionaries with exact IDs and span occurrences.

    Gold dictionaries may include ``utt``. If supplied, both gold and predicted
    spans are checked against it. Without raw text, schema integrity, exact
    boundaries and literal text equality still apply to every matching key.
    """
    gold_by_id = _contract_index(gold, "gold")
    predicted_by_id = _contract_index(predictions, "prediction")
    if not gold_by_id:
        raise ValueError("Cannot evaluate an empty gold split")
    missing = gold_by_id.keys() - predicted_by_id.keys()
    unexpected = predicted_by_id.keys() - gold_by_id.keys()
    if missing or unexpected:
        raise ValueError(f"Prediction IDs do not match gold IDs: missing={sorted(missing)!r}, unexpected={sorted(unexpected)!r}")
    correct = tp = fp = fn = 0
    for record_id, reference in gold_by_id.items():
        predicted = predicted_by_id[record_id]
        utt = reference.get("utt") if isinstance(reference, dict) else getattr(reference, "utt", None)
        if utt is not None and not isinstance(utt, str):
            raise ValueError("Raw gold utterance must be a string")
        reference_keys = _contract_span_keys(reference, utt)
        predicted_keys = _contract_span_keys(predicted, utt)
        correct += int(_contract_field(reference, "intent") == _contract_field(predicted, "intent"))
        tp += len(reference_keys & predicted_keys)
        fp += len(predicted_keys - reference_keys)
        fn += len(reference_keys - predicted_keys)
    denominator = 2 * tp + fp + fn
    return {"intent_accuracy": correct / len(gold_by_id),
            "slot_micro_f1": 2 * tp / denominator if denominator else 0.0,
            "slot_tp": tp, "slot_fp": fp, "slot_fn": fn,
            "num_examples": len(gold_by_id), "correct_intents": correct}


def bio_to_character_spans(utt, token_offsets, tags):
    """Decode BIO into original character coordinates without tokenization bias.

    Offsets accept ``(start,end)`` pairs, dictionaries, or objects with start/end.
    Orphan/type-changing I starts a new span, consistently for both architectures.
    """
    if not isinstance(utt, str):
        raise TypeError("Utterance must be a string")
    token_offsets, tags = list(token_offsets), list(tags)
    if len(token_offsets) != len(tags):
        raise ValueError("BIO tag count must equal unpadded token count")
    offsets, previous_end = [], 0
    for offset in token_offsets:
        if isinstance(offset, (tuple, list)) and len(offset) == 2:
            start, end = offset
        else:
            start, end = _contract_field(offset, "start"), _contract_field(offset, "end")
        if type(start) is not int or type(end) is not int or not previous_end <= start < end <= len(utt):
            raise ValueError("Token offsets must be ordered, nonoverlapping character slices")
        offsets.append((start, end))
        previous_end = end
    spans = []
    active_type = None
    active_start = active_end = 0

    def close():
        nonlocal active_type
        if active_type is not None:
            spans.append({"slot_type": active_type, "text": utt[active_start:active_end],
                          "start": active_start, "end": active_end})
        active_type = None

    for (start, end), tag in zip(offsets, tags):
        if not isinstance(tag, str):
            raise ValueError("BIO tags must be strings")
        if tag == "O":
            close()
            continue
        if len(tag) < 3 or tag[:2] not in {"B-", "I-"} or not tag[2:]:
            raise ValueError(f"Invalid BIO tag: {tag!r}")
        slot_type = tag[2:]
        if tag.startswith("B-") or active_type != slot_type:
            close()
            active_type, active_start = slot_type, start
        active_end = end
    close()
    return spans


## Environment

Install the repository's single `requirements.txt` in the local Python 3.12 / CUDA environment before opening this notebook. For a standalone copy, install the same pinned environment; the additional Transformer packages are pandas 2.2.3, gensim 4.3.3, SciPy 1.13.1, scikit-learn 1.6.1 and tqdm 4.67.1. No cell installs arbitrary latest packages.

This notebook preserves the original Transformer architecture and training settings, except for constructing all 111 BIO labels from the 55 training slot types. Stage 1 is executable below. Stage 2 is defined and counted but is not trained in the supplied experiment sequence. All numerical outputs were cleared after the evaluator/data change; fresh training is required before reporting results.


In [ ]:
from importlib.metadata import version
for distribution in ('torch', 'numpy', 'pandas', 'gensim', 'scipy', 'scikit-learn', 'tqdm'):
    print(distribution, version(distribution))


## 2. Imports and configuration

In [ ]:
import os
import re
import math
import random
import zipfile
import copy
import json
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from gensim.models import KeyedVectors

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.neighbors import NearestNeighbors

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

WORK_ROOT = Path.cwd()
NOTEBOOK_PATH = WORK_ROOT / "NLP_Final_project.ipynb"
RUN_ROOT = WORK_ROOT / "artifacts" / "transformer_v2"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_PATH = WORK_ROOT / "data" / "1.0" / "data" / "en-US.jsonl"
if not NOTEBOOK_PATH.is_file():
    raise FileNotFoundError("Keep NLP_Final_project.ipynb in the kernel working directory.")
notebook_document = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
current_code_hashes = [hashlib.sha256("".join(c["source"]).encode("utf-8")).hexdigest()
                       for c in notebook_document["cells"] if c["cell_type"] == "code"]
if current_code_hashes != notebook_document["metadata"]["evaluator_v2"]["code_cell_sha256"]:
    raise ValueError("Notebook source differs from its reviewed evaluator-v2 manifest")
NOTEBOOK_CODE_SHA256 = hashlib.sha256(json.dumps(current_code_hashes, separators=(",", ":")).encode("ascii")).hexdigest()

EXPECTED_SPLITS = {
    "train": 11514,
    "dev": 2033,
    "test": 2974,
}

MAX_LEN = 64
MAX_CHARS = 24

W2V_DIM = 300
W2V_ZIP = WORK_ROOT / "data" / "pretrained" / "nlpl_model_6.zip"
W2V_DIR = WORK_ROOT / "data" / "pretrained" / "nlpl_model_6"
W2V_URL = "https://vectors.nlpl.eu/repository/20/6.zip"

D_MODEL = 256
N_HEADS = 4
N_LAYERS = 4
D_FF = 768
DROPOUT = 0.15

CHAR_DIM = 24
CHAR_CHANNELS = 32
CHAR_KERNELS = (3, 4, 5)

BATCH_SIZE = 64
EPOCHS = 12
PATIENCE = 3
LR = 3e-4
WEIGHT_DECAY = 1e-4

LAMBDA_INTENT = 1.0
LAMBDA_SLOT = 1.0

print("Config loaded.")


## Load and verify the identical original MASSIVE release

Both notebooks use the same inline loader, English JSONL SHA-256 and original split checks. The loader downloads the official archive only if the pinned English file is absent. An existing file with different bytes is rejected. Uploaded CSVs are no longer accepted as an unverified alternate source.


In [ ]:
verified_splits, dataset_manifest = ensure_verified_massive(DATASET_PATH)
assert dataset_manifest['dataset_sha256'] == DATASET_SHA256
df = pd.DataFrame([dict(row, id=str(row['id']))
                   for part in ('train', 'dev', 'test') for row in verified_splits[part]])
required = {'id', 'partition', 'utt', 'intent', 'annot_utt'}
assert required.issubset(df.columns)
assert not df['id'].duplicated().any()
assert {part: int((df['partition'] == part).sum()) for part in EXPECTED_COUNTS} == EXPECTED_COUNTS
GOLD_BY_ID = {str(row.id): {'record_id': str(row.id), 'utt': row.utt,
               'intent': row.intent, 'slots': parse_annotated_spans(row.utt, row.annot_utt)}
              for row in df.itertuples(index=False)}
(RUN_ROOT / 'dataset_manifest.json').write_text(json.dumps(dataset_manifest, indent=2) + '\n', encoding='utf-8')
print('VERIFIED_DATASET_MANIFEST:', json.dumps(dataset_manifest, indent=2))
print('EVALUATOR_VERSION:', EVALUATOR_VERSION)
print('Dataset identity and every original partition verified.')


## 4. Tokenization and BIO parsing

In [ ]:
TOKEN_RE = re.compile(r"\w+(?:['\u2019\-]\w+)*|[^\w\s]", re.UNICODE)

def tokenize_with_offsets(text):
    return [(match.group(0), match.start(), match.end()) for match in TOKEN_RE.finditer(text)]

def tokenize(text):
    return [token for token, _, _ in tokenize_with_offsets(text)]

def is_lexical(token):
    return bool(re.search(r"[A-Za-z0-9]", token))

def parse_row_targets(utt, annot_utt):
    # Predictive token text and positions come exclusively from raw utt.
    pieces = tokenize_with_offsets(utt)
    tokens = [token for token, _, _ in pieces]
    offsets = [(start, end) for _, start, end in pieces]
    spans = parse_annotated_spans(utt, annot_utt)
    bio = ['O'] * len(tokens)
    slot_mask = [False] * len(tokens)
    for span in spans:
        covered = [i for i, (start, end) in enumerate(offsets)
                   if start < span['end'] and span['start'] < end]
        if not covered or offsets[covered[0]][0] != span['start'] or offsets[covered[-1]][1] != span['end']:
            raise ValueError('A gold boundary is incompatible with the fixed Transformer tokenizer')
        for j, i in enumerate(covered):
            if bio[i] != 'O':
                raise ValueError('Overlapping annotation spans')
            bio[i] = ('B-' if j == 0 else 'I-') + span['slot_type']
            slot_mask[i] = True
    if bio_to_character_spans(utt, offsets, bio) != spans:
        raise ValueError('Gold BIO annotations do not round-trip to exact original character spans')
    return {'tokens': tokens, 'offsets': offsets, 'bio': bio,
            'slot_mask': slot_mask, 'gold_spans': spans}

for row in tqdm(df.itertuples(index=False), total=len(df), desc='Exact annotation alignment'):
    parse_row_targets(row.utt, row.annot_utt)
max_observed = max(len(tokenize(x)) for x in df['utt'])
assert max_observed <= MAX_LEN
print('All original gold boundaries round-trip; maximum token length:', max_observed)


## 5. Build task labels and vocabularies using train only

In [ ]:
train_df = df[df["partition"] == "train"].copy()
dev_df = df[df["partition"] == "dev"].copy()
test_df = df[df["partition"] == "test"].copy()

intent_labels = sorted(train_df["intent"].unique())
intent2id = {x: i for i, x in enumerate(intent_labels)}
id2intent = {i: x for x, i in intent2id.items()}

slot_types = sorted({span['slot_type'] for row in train_df.itertuples(index=False)
                     for span in parse_annotated_spans(row.utt, row.annot_utt)})
assert len(slot_types) == 55
slot_labels = ['O'] + [prefix + '-' + slot_type for slot_type in slot_types for prefix in ('B', 'I')]
assert len(slot_labels) == 111
slot2id = {x: i for i, x in enumerate(slot_labels)}
id2slot = {i: x for x, i in slot2id.items()}

train_tokens_original = []
train_lexical_casefold = set()
char_counter = Counter()

for row in tqdm(train_df.itertuples(index=False), total=len(train_df), desc="Train vocab"):
    parsed = parse_row_targets(row.utt, row.annot_utt)
    for tok in parsed["tokens"]:
        train_tokens_original.append(tok)
        if is_lexical(tok):
            train_lexical_casefold.add(tok.casefold())
        char_counter.update(tok)

for split_name, split_df in [("dev", dev_df), ("test", test_df)]:
    unseen_intents = set(split_df["intent"]) - set(intent2id)
    assert not unseen_intents

    unseen_slot_labels = set()
    for row in split_df.itertuples(index=False):
        unseen_slot_labels.update(set(parse_row_targets(row.utt, row.annot_utt)['bio']) - set(slot2id))
    assert not unseen_slot_labels, f"{split_name}: {unseen_slot_labels}"

word_counter = Counter(tok.casefold() for tok in train_tokens_original)

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

word_vocab = [PAD_TOKEN, UNK_TOKEN] + sorted(word_counter)
word2id = {w: i for i, w in enumerate(word_vocab)}
id2word = {i: w for w, i in word2id.items()}

PAD_ID = word2id[PAD_TOKEN]
UNK_ID = word2id[UNK_TOKEN]

CHAR_PAD = "<CPAD>"
CHAR_UNK = "<CUNK>"

char_vocab = [CHAR_PAD, CHAR_UNK] + sorted(char_counter)
char2id = {c: i for i, c in enumerate(char_vocab)}

CHAR_PAD_ID = char2id[CHAR_PAD]
CHAR_UNK_ID = char2id[CHAR_UNK]

print("Intent labels:", len(intent_labels))
print("BIO labels:", len(slot_labels))
print("Word vocab:", len(word_vocab))
print("Char vocab:", len(char_vocab))
print("Embedding params:", len(word_vocab) * W2V_DIM)


## 6. Download/load Wikipedia Word2Vec 2017

In [ ]:
def download_file(url, destination):
    """Stream to a temporary file; publish only a complete new cache entry."""
    import os
    import tempfile
    from urllib.request import Request, urlopen

    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() or destination.is_symlink():
        if destination.is_symlink() or not destination.is_file():
            raise ValueError('Expected a regular pretrained-vector cache file: ' + str(destination))
        return 0
    descriptor, temporary_name = tempfile.mkstemp(
        prefix='.' + destination.name + '-', suffix='.part', dir=destination.parent)
    temporary = Path(temporary_name)
    try:
        request = Request(url, headers={'User-Agent': 'SC4002-Transformer-vector-cache/2'})
        with os.fdopen(descriptor, 'wb') as output:
            with urlopen(request, timeout=60) as response:
                expected = response.headers.get('Content-Length')
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    output.write(chunk)
            output.flush()
            os.fsync(output.fileno())
        observed = temporary.stat().st_size
        if not observed or (expected is not None and observed != int(expected)):
            raise ValueError('Incomplete pretrained-vector download')
        # Both files are in the same directory. Hard-link creation is atomic and
        # refuses an existing destination, including one created concurrently.
        os.link(temporary, destination)
        return 0
    except Exception as error:
        print('Pretrained-vector download failed:', str(error))
        print('Download the original NLPL Model 6 ZIP manually and place it at:', destination)
        return 1
    finally:
        temporary.unlink(missing_ok=True)

if not W2V_ZIP.exists():
    rc = download_file(W2V_URL, W2V_ZIP)

    invalid = (
        rc != 0
        or not W2V_ZIP.exists()
        or W2V_ZIP.stat().st_size < 1_000_000
    )

    if invalid:
        raise FileNotFoundError('Place the original NLPL Model 6 ZIP at ' + str(W2V_ZIP)
                                + '; automatic download failed.')

W2V_DIR.mkdir(parents=True, exist_ok=True)

if not any(W2V_DIR.iterdir()):
    with zipfile.ZipFile(W2V_ZIP, "r") as zf:
        zf.extractall(W2V_DIR)

def candidate_vector_files(root):
    exts = {".txt", ".bin", ".vec", ".kv", ".model"}
    files = [
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in exts
    ]
    return sorted(files, key=lambda p: p.stat().st_size, reverse=True)

def try_load(path):
    try:
        return KeyedVectors.load(str(path), mmap="r")
    except Exception:
        pass

    order = [True, False] if path.suffix.lower() == ".bin" else [False, True]

    for binary in order:
        try:
            return KeyedVectors.load_word2vec_format(
                str(path),
                binary=binary,
                unicode_errors="ignore",
            )
        except Exception:
            pass

    return None

wv = None
loaded_from = None

for p in candidate_vector_files(W2V_DIR):
    obj = try_load(p)
    if obj is not None:
        wv = obj
        loaded_from = p
        break

assert wv is not None
assert wv.vector_size == W2V_DIM

print("Loaded:", loaded_from)
print("Dimension:", wv.vector_size)
print("Pretrained vocab:", len(wv))


## 7. Case-insensitive pretrained lookup

In [ ]:
POS_SUFFIX_RE = re.compile(
    r"_(ADJ|ADP|ADV|AUX|CCONJ|DET|INTJ|NOUN|NUM|PART|PRON|PROPN|PUNCT|SCONJ|SYM|VERB|X)$"
)

probe_n = min(50_000, len(wv))

pos_ratio = np.mean([
    bool(POS_SUFFIX_RE.search(k))
    for k in wv.index_to_key[:probe_n]
])

print("POS-suffix-like ratio:", f"{pos_ratio:.2%}")

def strip_pos_suffix(k):
    return POS_SUFFIX_RE.sub("", k)

cf_sums = {}
cf_counts = Counter()

for key in tqdm(wv.index_to_key, desc="Pretrained casefold map"):
    surface = strip_pos_suffix(key) if pos_ratio > 0.20 else key
    cf = surface.casefold()
    vec = wv[key].astype(np.float32)

    if cf not in cf_sums:
        cf_sums[cf] = vec.copy()
    else:
        cf_sums[cf] += vec

    cf_counts[cf] += 1

casefold_vectors = {
    k: cf_sums[k] / cf_counts[k]
    for k in cf_sums
}

pretrained_casefold_vocab = set(casefold_vectors)

print("Casefold pretrained vocab:", len(pretrained_casefold_vocab))

## 8. Fair embedding initialization

In [ ]:
rng = np.random.default_rng(SEED)

embedding_matrix = rng.normal(
    0.0,
    0.05,
    size=(len(word_vocab), W2V_DIM),
).astype(np.float32)

embedding_matrix[PAD_ID] = 0.0
embedding_matrix[UNK_ID] = rng.normal(
    0.0,
    0.05,
    size=(W2V_DIM,),
).astype(np.float32)

pretrained_initialized = 0
random_initialized = 0

for word, idx in tqdm(word2id.items(), desc="Embedding init"):
    if idx < 2:
        continue

    if word in pretrained_casefold_vocab:
        embedding_matrix[idx] = casefold_vectors[word]
        pretrained_initialized += 1
    else:
        random_initialized += 1

print("Pretrained initialized:", pretrained_initialized)
print("Unique random train-vocab entries:", random_initialized)
print(
    "Pretrained-init coverage:",
    f"{100*pretrained_initialized/max(pretrained_initialized+random_initialized,1):.2f}%"
)

## 9. Build Train-OOV and Pretrained-OOV metadata

In [ ]:
def build_oov_metadata(split_df):
    example_rows = []

    for row in tqdm(
        split_df.itertuples(index=False),
        total=len(split_df),
        desc=f"OOV metadata {split_df['partition'].iloc[0]}"
    ):
        parsed = parse_row_targets(row.utt, row.annot_utt)
        tokens = parsed["tokens"]
        slot_mask = parsed["slot_mask"]

        train_oov_flags = []
        pretrained_oov_flags = []

        for tok in tokens:
            lexical = is_lexical(tok)
            cf = tok.casefold()

            train_oov_flags.append(
                lexical and cf not in train_lexical_casefold
            )

            pretrained_oov_flags.append(
                lexical and cf not in pretrained_casefold_vocab
            )

        has_train_oov = any(train_oov_flags)
        has_slot_train_oov = any(
            o and s for o, s in zip(train_oov_flags, slot_mask)
        )

        if not has_train_oov:
            train_group = "No-Train-OOV"
        elif has_slot_train_oov:
            train_group = "Slot-Train-OOV"
        else:
            train_group = "NonSlot-Train-OOV"

        has_pre_oov = any(pretrained_oov_flags)
        has_slot_pre_oov = any(
            o and s for o, s in zip(pretrained_oov_flags, slot_mask)
        )

        if not has_pre_oov:
            pretrained_group = "No-Pretrained-OOV"
        elif has_slot_pre_oov:
            pretrained_group = "Slot-Pretrained-OOV"
        else:
            pretrained_group = "NonSlot-Pretrained-OOV"

        example_rows.append({
            "id": row.id,
            "partition": row.partition,
            "n_tokens": len(tokens),
            "n_gold_slots": sum(tag.startswith("B-") for tag in parsed["bio"]),
            "n_train_oov": sum(train_oov_flags),
            "n_slot_train_oov": sum(
                o and s for o, s in zip(train_oov_flags, slot_mask)
            ),
            "train_oov_group": train_group,
            "pretrained_oov_group": pretrained_group,
        })

    return pd.DataFrame(example_rows)

dev_oov = build_oov_metadata(dev_df)
test_oov = build_oov_metadata(test_df)

print("DEV Train-OOV groups")
display(dev_oov["train_oov_group"].value_counts().to_frame("n"))

print("TEST Train-OOV groups")
display(test_oov["train_oov_group"].value_counts().to_frame("n"))

print("DEV Pretrained-OOV groups")
display(dev_oov["pretrained_oov_group"].value_counts().to_frame("n"))


## 10. Dataset and collate functions

In [ ]:
class MassiveDataset(Dataset):
    def __init__(self, frame, include_chars=False):
        self.rows = []
        self.include_chars = include_chars

        for row in frame.itertuples(index=False):
            parsed = parse_row_targets(row.utt, row.annot_utt)
            tokens = parsed["tokens"]
            bio = parsed["bio"]

            word_ids = [
                word2id.get(tok.casefold(), UNK_ID)
                for tok in tokens
            ]

            slot_ids = [
                slot2id[tag]
                for tag in bio
            ]

            char_ids = []

            if include_chars:
                for tok in tokens:
                    ids = [
                        char2id.get(c, CHAR_UNK_ID)
                        for c in tok[:MAX_CHARS]
                    ]
                    ids += [CHAR_PAD_ID] * (MAX_CHARS - len(ids))
                    char_ids.append(ids)

            self.rows.append({
                "id": str(row.id),
                "utt": row.utt,
                "offsets": parsed['offsets'],
                "word_ids": word_ids,
                "slot_ids": slot_ids,
                "intent_id": intent2id[row.intent],
                "char_ids": char_ids,
            })

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


def make_collate(include_chars=False):
    def collate(batch):
        B = len(batch)
        T = max(len(x["word_ids"]) for x in batch)

        word_ids = torch.full(
            (B, T), PAD_ID, dtype=torch.long
        )

        slot_ids = torch.full(
            (B, T), -100, dtype=torch.long
        )

        attention_mask = torch.zeros(
            (B, T), dtype=torch.bool
        )

        intent_ids = torch.tensor(
            [x["intent_id"] for x in batch],
            dtype=torch.long,
        )

        char_ids = None

        if include_chars:
            char_ids = torch.full(
                (B, T, MAX_CHARS),
                CHAR_PAD_ID,
                dtype=torch.long,
            )

        ids = []

        for i, x in enumerate(batch):
            n = len(x["word_ids"])

            word_ids[i, :n] = torch.tensor(x["word_ids"])
            slot_ids[i, :n] = torch.tensor(x["slot_ids"])
            attention_mask[i, :n] = True

            if include_chars:
                char_ids[i, :n] = torch.tensor(x["char_ids"])

            ids.append(x["id"])

        return {
            "ids": ids,
            "utts": [x['utt'] for x in batch],
            "token_offsets": [x['offsets'] for x in batch],
            "word_ids": word_ids,
            "slot_ids": slot_ids,
            "attention_mask": attention_mask,
            "intent_ids": intent_ids,
            "char_ids": char_ids,
        }

    return collate


## 11. Stage-1 and Stage-2 model definitions

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0, max_len, dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class JointTransformerStage1(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            len(word_vocab),
            W2V_DIM,
            padding_idx=PAD_ID,
        )

        self.embedding.weight.data.copy_(
            torch.tensor(embedding_matrix)
        )

        self.proj = nn.Linear(W2V_DIM, D_MODEL)
        self.pos = PositionalEncoding(D_MODEL)

        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dim_feedforward=D_FF,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=N_LAYERS,
            norm=nn.LayerNorm(D_MODEL),
        )

        self.intent_head = nn.Linear(
            D_MODEL,
            len(intent_labels),
        )

        self.slot_head = nn.Linear(
            D_MODEL,
            len(slot_labels),
        )

    def forward(self, word_ids, attention_mask, char_ids=None):
        x = self.embedding(word_ids)
        x = self.proj(x)
        x = self.pos(x)

        x = self.encoder(
            x,
            src_key_padding_mask=~attention_mask,
        )

        mask = attention_mask.unsqueeze(-1).float()
        pooled = (x * mask).sum(1) / mask.sum(1).clamp_min(1.0)

        return self.intent_head(pooled), self.slot_head(x)


class CharCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.char_embedding = nn.Embedding(
            len(char_vocab),
            CHAR_DIM,
            padding_idx=CHAR_PAD_ID,
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                CHAR_DIM,
                CHAR_CHANNELS,
                kernel_size=k,
            )
            for k in CHAR_KERNELS
        ])

        self.output_dim = CHAR_CHANNELS * len(CHAR_KERNELS)

    def forward(self, char_ids):
        B, T, C = char_ids.shape

        x = char_ids.reshape(B * T, C)
        x = self.char_embedding(x).transpose(1, 2)

        outs = []

        for conv in self.convs:
            h = torch.relu(conv(x))
            h = torch.max(h, dim=-1).values
            outs.append(h)

        out = torch.cat(outs, dim=-1)
        return out.reshape(B, T, -1)


class JointTransformerStage2(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            len(word_vocab),
            W2V_DIM,
            padding_idx=PAD_ID,
        )

        self.embedding.weight.data.copy_(
            torch.tensor(embedding_matrix)
        )

        self.char_encoder = CharCNN()

        self.proj = nn.Linear(
            W2V_DIM + self.char_encoder.output_dim,
            D_MODEL,
        )

        self.pos = PositionalEncoding(D_MODEL)

        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dim_feedforward=D_FF,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=N_LAYERS,
            norm=nn.LayerNorm(D_MODEL),
        )

        self.intent_head = nn.Linear(
            D_MODEL,
            len(intent_labels),
        )

        self.slot_head = nn.Linear(
            D_MODEL,
            len(slot_labels),
        )

    def forward(self, word_ids, attention_mask, char_ids):
        word_x = self.embedding(word_ids)
        char_x = self.char_encoder(char_ids)

        x = torch.cat([word_x, char_x], dim=-1)
        x = self.proj(x)
        x = self.pos(x)

        x = self.encoder(
            x,
            src_key_padding_mask=~attention_mask,
        )

        mask = attention_mask.unsqueeze(-1).float()
        pooled = (x * mask).sum(1) / mask.sum(1).clamp_min(1.0)

        return self.intent_head(pooled), self.slot_head(x)

## 12. Parameter counts

In [ ]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
    return total, trainable

for name, model in [
    ("Stage 1", JointTransformerStage1()),
    ("Stage 2", JointTransformerStage2()),
]:
    total, trainable = count_params(model)

    print(
        name,
        "| total:", f"{total:,}",
        "| trainable:", f"{trainable:,}",
        "| M:", f"{total/1e6:.3f}",
    )

    assert total <= 20_000_000
    assert total == {'Stage 1': 4_345_243, 'Stage 2': 4_379_947}[name]


## Shared exact-character-span evaluation

The same source used by the BiLSTM notebook is inlined at the start. A correct slot requires the exact slot type and original utterance character start/end offsets; repeated text at different positions is scored separately. Character positions avoid dependence on either model's tokenizer. Scores pool TP/FP/FN over the complete ID-checked split. Invalid source slices, missing/extra/duplicate IDs, malformed BIO and unequal sequence lengths fail rather than silently changing the denominator.


In [ ]:
def score_prediction_frame(frame, repeated_diagnostic_draws=False):
    gold, predicted = [], []
    for draw, row in enumerate(frame.itertuples(index=False)):
        expected = GOLD_BY_ID[str(row.id)]
        observed = dict(row.prediction)
        target = dict(expected)
        if repeated_diagnostic_draws:
            # Nearest-neighbor matching samples with replacement. Each diagnostic
            # draw is explicit; full-split evaluation never relaxes duplicate IDs.
            draw_id = str(row.id) + '#diagnostic_draw:' + str(draw)
            target['record_id'] = observed['record_id'] = draw_id
        gold.append(target)
        predicted.append(observed)
    return score_exact_spans(gold, predicted)

identity = score_exact_spans(list(GOLD_BY_ID.values()), list(GOLD_BY_ID.values()))
assert identity['intent_accuracy'] == 1.0 and identity['slot_micro_f1'] == 1.0
print('Shared evaluator exact-span self-check passed:', EVALUATOR_VERSION)


## 14. Dataloaders

In [ ]:
train_s1 = MassiveDataset(train_df, include_chars=False)
dev_s1 = MassiveDataset(dev_df, include_chars=False)
test_s1 = MassiveDataset(test_df, include_chars=False)

train_s2 = MassiveDataset(train_df, include_chars=True)
dev_s2 = MassiveDataset(dev_df, include_chars=True)
test_s2 = MassiveDataset(test_df, include_chars=True)

train_loader_s1 = DataLoader(
    train_s1,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=make_collate(False),
)

dev_loader_s1 = DataLoader(
    dev_s1,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=make_collate(False),
)

test_loader_s1 = DataLoader(
    test_s1,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=make_collate(False),
)

train_loader_s2 = DataLoader(
    train_s2,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=make_collate(True),
)

dev_loader_s2 = DataLoader(
    dev_s2,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=make_collate(True),
)

test_loader_s2 = DataLoader(
    test_s2,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=make_collate(True),
)

## 15. Training/evaluation functions

In [ ]:
intent_loss_fn = nn.CrossEntropyLoss()
slot_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)


def move_batch(batch, include_chars):
    out = {
        "ids": batch["ids"],
        "word_ids": batch["word_ids"].to(DEVICE),
        "slot_ids": batch["slot_ids"].to(DEVICE),
        "attention_mask": batch["attention_mask"].to(DEVICE),
        "intent_ids": batch["intent_ids"].to(DEVICE),
        "char_ids": None,
    }

    if include_chars:
        out["char_ids"] = batch["char_ids"].to(DEVICE)

    return out


def forward_loss(model, batch, include_chars):
    intent_logits, slot_logits = model(
        batch["word_ids"],
        batch["attention_mask"],
        batch["char_ids"],
    )

    loss_intent = intent_loss_fn(
        intent_logits,
        batch["intent_ids"],
    )

    loss_slot = slot_loss_fn(
        slot_logits.reshape(-1, len(slot_labels)),
        batch["slot_ids"].reshape(-1),
    )

    loss = (
        LAMBDA_INTENT * loss_intent
        + LAMBDA_SLOT * loss_slot
    )

    return loss, intent_logits, slot_logits


@torch.no_grad()
def evaluate_model(model, loader, include_chars):
    model.eval()
    losses, rows = [], []
    for raw_batch in loader:
        batch = move_batch(raw_batch, include_chars)
        loss, intent_logits, slot_logits = forward_loss(model, batch, include_chars)
        losses.append(loss.item())
        pred_int = intent_logits.argmax(-1).cpu().tolist()
        pred_slot = slot_logits.argmax(-1).cpu().tolist()
        for i, record_id in enumerate(raw_batch['ids']):
            n = int(raw_batch['attention_mask'][i].sum().item())
            tags = [id2slot[index] for index in pred_slot[i][:n]]
            utterance = raw_batch['utts'][i]
            prediction = {'record_id': str(record_id), 'intent': id2intent[pred_int[i]],
                          'utt': utterance,
                          'slots': bio_to_character_spans(utterance, raw_batch['token_offsets'][i], tags)}
            expected = GOLD_BY_ID[str(record_id)]
            rows.append({'id': str(record_id), 'gold_intent_id': intent2id[expected['intent']],
                         'pred_intent_id': pred_int[i], 'prediction': prediction})
    frame = pd.DataFrame(rows)
    metrics = score_prediction_frame(frame)
    expected_ids = [row['id'] for row in loader.dataset.rows]
    if frame['id'].tolist() != expected_ids:
        raise ValueError('Evaluation must cover the complete original split in order')
    return {'loss': float(np.mean(losses)), 'metrics': metrics,
            'intent_accuracy': 100.0 * metrics['intent_accuracy'],
            'slot_micro_f1': 100.0 * metrics['slot_micro_f1'],
            'joint_score': 50.0 * (metrics['intent_accuracy'] + metrics['slot_micro_f1']),
            'pred_frame': frame, 'evaluator_version': EVALUATOR_VERSION}


def train_model(
    model,
    train_loader,
    dev_loader,
    include_chars,
    checkpoint_path,
):
    seed_everything()
    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    if checkpoint_path.exists():
        raise FileExistsError('Use a fresh run directory; existing checkpoint is preserved: ' + str(checkpoint_path))
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_score = -1.0
    patience_left = PATIENCE
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        for raw_batch in tqdm(
            train_loader,
            desc=f"Epoch {epoch}",
            leave=False,
        ):
            batch = move_batch(raw_batch, include_chars)

            optimizer.zero_grad()

            loss, _, _ = forward_loss(
                model,
                batch,
                include_chars,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )

            optimizer.step()
            train_losses.append(loss.item())

        dev_result = evaluate_model(
            model,
            dev_loader,
            include_chars,
        )

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "dev_loss": dev_result["loss"],
            "dev_intent_acc": dev_result["intent_accuracy"],
            "dev_slot_f1": dev_result["slot_micro_f1"],
            "dev_joint": dev_result["joint_score"],
        }

        history.append(row)

        print(
            f"Epoch {epoch:02d} | "
            f"train {row['train_loss']:.4f} | "
            f"dev {row['dev_loss']:.4f} | "
            f"intent {row['dev_intent_acc']:.2f} | "
            f"slot {row['dev_slot_f1']:.2f} | "
            f"joint {row['dev_joint']:.2f}"
        )

        if row["dev_joint"] > best_score:
            best_score = row["dev_joint"]
            patience_left = PATIENCE

            torch.save(
                {
                    "model_state_dict": copy.deepcopy(model.state_dict()),
                    "epoch": epoch,
                    "dev_joint": best_score,
                    "evaluator_version": EVALUATOR_VERSION,
                    "dataset_sha256": DATASET_SHA256,
                    "dataset_manifest": dataset_manifest,
                    "notebook_code_sha256": NOTEBOOK_CODE_SHA256,
                    "dev_metrics": dev_result['metrics'],
                },
                checkpoint_path,
            )
        else:
            patience_left -= 1

            if patience_left <= 0:
                print("Early stopping.")
                break

    ckpt = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=True,
    )

    if (ckpt['evaluator_version'] != EVALUATOR_VERSION
            or ckpt['dataset_sha256'] != DATASET_SHA256
            or ckpt['notebook_code_sha256'] != NOTEBOOK_CODE_SHA256):
        raise ValueError('Checkpoint belongs to a different scoring/data/code contract')
    model.load_state_dict(ckpt['model_state_dict'])
    reloaded = evaluate_model(model, dev_loader, include_chars)
    if reloaded['metrics'] != ckpt['dev_metrics']:
        raise ValueError('Reloaded checkpoint does not reproduce selected development metrics')

    print(
        "Best checkpoint:",
        checkpoint_path,
        "| epoch", ckpt["epoch"],
        "| dev joint", f"{ckpt['dev_joint']:.2f}",
    )

    return model, pd.DataFrame(history)


# Stage 1
## 16. Train Word2Vec-initialized shared Transformer

In [ ]:
STAGE1_CKPT = RUN_ROOT / "stage1_best.pt"

seed_everything(SEED)

stage1_model = JointTransformerStage1()

stage1_model, stage1_history = train_model(
    stage1_model,
    train_loader_s1,
    dev_loader_s1,
    include_chars=False,
    checkpoint_path=STAGE1_CKPT,
)

display(stage1_history)


## 17. Stage-1 overall dev result

In [ ]:
stage1_dev = evaluate_model(
    stage1_model,
    dev_loader_s1,
    include_chars=False,
)

print(
    "Stage 1 DEV |",
    "Intent:", f"{stage1_dev['intent_accuracy']:.2f}",
    "| Slot F1:", f"{stage1_dev['slot_micro_f1']:.2f}",
    "| Joint:", f"{stage1_dev['joint_score']:.2f}",
)

# E1
## 18. Stage-1 performance by Train-OOV subgroup

In [ ]:
def merge_oov(pred_frame, oov_frame):
    return pred_frame.merge(
        oov_frame,
        on="id",
        how="left",
        validate="one_to_one",
    )

def subset_metrics(frame, repeated_diagnostic_draws=False):
    metrics = score_prediction_frame(frame, repeated_diagnostic_draws=repeated_diagnostic_draws)
    return {'n_examples': metrics['num_examples'],
            'intent_accuracy': 100.0 * metrics['intent_accuracy'],
            'slot_micro_f1': 100.0 * metrics['slot_micro_f1']}


s1_dev = merge_oov(
    stage1_dev["pred_frame"],
    dev_oov,
)

e1_rows = []

for group in [
    "No-Train-OOV",
    "NonSlot-Train-OOV",
    "Slot-Train-OOV",
]:
    sub = s1_dev[
        s1_dev["train_oov_group"] == group
    ]

    if len(sub):
        e1_rows.append({
            "group": group,
            **subset_metrics(sub),
        })

E1_table = pd.DataFrame(e1_rows)

display(
    E1_table.style.format({
        "intent_accuracy": "{:.2f}",
        "slot_micro_f1": "{:.2f}",
    })
)

f1_no = float(
    E1_table.loc[
        E1_table["group"] == "No-Train-OOV",
        "slot_micro_f1",
    ].iloc[0]
)

f1_slot = float(
    E1_table.loc[
        E1_table["group"] == "Slot-Train-OOV",
        "slot_micro_f1",
    ].iloc[0]
)

raw_gap = f1_no - f1_slot

print(
    "E1 raw gap:",
    f"{raw_gap:+.2f} points"
)


## 19. Secondary E1: Pretrained-OOV

In [ ]:
pre_rows = []

for group in [
    "No-Pretrained-OOV",
    "NonSlot-Pretrained-OOV",
    "Slot-Pretrained-OOV",
]:
    sub = s1_dev[
        s1_dev["pretrained_oov_group"] == group
    ]

    if len(sub):
        pre_rows.append({
            "group": group,
            **subset_metrics(sub),
        })

E1_pretrained_table = pd.DataFrame(pre_rows)

display(
    E1_pretrained_table.style.format({
        "intent_accuracy": "{:.2f}",
        "slot_micro_f1": "{:.2f}",
    })
)

# E2
## 20. Exact difficulty strata

In [ ]:
def length_bucket(n):
    if n <= 4:
        return "01-04"
    if n <= 8:
        return "05-08"
    if n <= 12:
        return "09-12"
    if n <= 16:
        return "13-16"
    return "17+"

s1_control = s1_dev.copy()
s1_control["length_bucket"] = s1_control["n_tokens"].map(length_bucket)

strata_rows = []

for (length_b, n_slots), grp in s1_control.groupby(
    ["length_bucket", "n_gold_slots"]
):
    no = grp[
        grp["train_oov_group"] == "No-Train-OOV"
    ]

    slot = grp[
        grp["train_oov_group"] == "Slot-Train-OOV"
    ]

    if len(no) >= 10 and len(slot) >= 10:
        m_no = subset_metrics(no)
        m_slot = subset_metrics(slot)

        strata_rows.append({
            "length_bucket": length_b,
            "n_gold_slots": n_slots,
            "n_no_oov": len(no),
            "n_slot_oov": len(slot),
            "no_oov_f1": m_no["slot_micro_f1"],
            "slot_oov_f1": m_slot["slot_micro_f1"],
            "gap": m_no["slot_micro_f1"] - m_slot["slot_micro_f1"],
        })

E2_strata = pd.DataFrame(strata_rows)

display(
    E2_strata.style.format({
        "no_oov_f1": "{:.2f}",
        "slot_oov_f1": "{:.2f}",
        "gap": "{:+.2f}",
    })
)

if len(E2_strata):
    print(
        "Median controlled gap:",
        f"{E2_strata['gap'].median():+.2f}"
    )

## 21. Nearest-neighbor matched control

In [ ]:
def matched_control(frame):
    slot_oov = frame[
        frame["train_oov_group"] == "Slot-Train-OOV"
    ].copy()

    no_oov = frame[
        frame["train_oov_group"] == "No-Train-OOV"
    ].copy()

    features = ["n_tokens", "n_gold_slots"]

    X_slot = slot_oov[features].astype(float).to_numpy()
    X_no = no_oov[features].astype(float).to_numpy()

    pooled = np.vstack([X_slot, X_no])
    sd = pooled.std(axis=0)
    sd[sd == 0] = 1.0

    nn = NearestNeighbors(
        n_neighbors=1,
        metric="euclidean",
    )

    nn.fit(X_no / sd)
    distances, indices = nn.kneighbors(X_slot / sd)

    matched_no = no_oov.iloc[
        indices[:, 0]
    ].copy().reset_index(drop=True)

    matched_slot = slot_oov.reset_index(drop=True)

    return matched_no, matched_slot, distances[:, 0]

matched_no, matched_slot, match_dist = matched_control(
    s1_dev
)

matched_table = pd.DataFrame([
    {
        "group": "Matched No-Train-OOV",
        **subset_metrics(matched_no, repeated_diagnostic_draws=True),
        "avg_tokens": matched_no["n_tokens"].mean(),
        "avg_slots": matched_no["n_gold_slots"].mean(),
    },
    {
        "group": "Slot-Train-OOV",
        **subset_metrics(matched_slot),
        "avg_tokens": matched_slot["n_tokens"].mean(),
        "avg_slots": matched_slot["n_gold_slots"].mean(),
    },
])

display(
    matched_table.style.format({
        "intent_accuracy": "{:.2f}",
        "slot_micro_f1": "{:.2f}",
        "avg_tokens": "{:.2f}",
        "avg_slots": "{:.2f}",
    })
)

matched_gap = (
    matched_table.loc[
        matched_table["group"] == "Matched No-Train-OOV",
        "slot_micro_f1",
    ].iloc[0]
    -
    matched_table.loc[
        matched_table["group"] == "Slot-Train-OOV",
        "slot_micro_f1",
    ].iloc[0]
)

print(
    "Matched controlled gap:",
    f"{matched_gap:+.2f} points"
)
